In [2]:
from pathlib import Path
import pandas as pd

In [3]:
data_path = Path("../data/raw")

csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files:")

for file in csv_files:
    print(file)

Found 3 CSV files:
..\data\raw\JC-202608-citibike-tripdata.csv
..\data\raw\JC-202608-citibike-tripdata.csv\JC-202608-citibike-tripdata.csv
..\data\raw\JC-202608-citibike-tripdata.csv\__MACOSX\._JC-202608-citibike-tripdata.csv


In [4]:
csv_files = [
    path
    for path in data_path.rglob("*.csv")
    if path.is_file()
    and "__MACOSX" not in path.parts
    and not path.name.startswith("._")
]


In [5]:
df = pd.read_csv(
    csv_files[0],
    nrows=10_000
)

In [6]:
df.shape

(10000, 13)

In [7]:
df.columns.tolist()

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual']

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ride_id             10000 non-null  object 
 1   rideable_type       10000 non-null  object 
 2   started_at          10000 non-null  object 
 3   ended_at            10000 non-null  object 
 4   start_station_name  10000 non-null  object 
 5   start_station_id    10000 non-null  object 
 6   end_station_name    10000 non-null  object 
 7   end_station_id      9993 non-null   object 
 8   start_lat           10000 non-null  float64
 9   start_lng           10000 non-null  float64
 10  end_lat             9993 non-null   float64
 11  end_lng             9993 non-null   float64
 12  member_casual       10000 non-null  object 
dtypes: float64(4), object(9)
memory usage: 1015.8+ KB


# Examine missing data

In [13]:
missing = (
    df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing

end_station_id        0.07
end_lat               0.07
end_lng               0.07
ride_id               0.00
rideable_type         0.00
started_at            0.00
ended_at              0.00
start_station_name    0.00
start_station_id      0.00
end_station_name      0.00
start_lat             0.00
start_lng             0.00
member_casual         0.00
dtype: float64

# Check Duplicate

In [14]:
df["ride_id"].duplicated().sum()

0

In [16]:
df["ride_id"].nunique()

10000

In [17]:
df["rideable_type"].value_counts(dropna=False)

rideable_type
electric_bike    6169
classic_bike     3831
Name: count, dtype: int64

In [18]:
df["member_casual"].value_counts(dropna=False)

member_casual
member    7044
casual    2956
Name: count, dtype: int64

In [19]:
df["started_at"] = pd.to_datetime(
    df["started_at"],
    errors="coerce"
)

df["ended_at"] = pd.to_datetime(
    df["ended_at"],
    errors="coerce"
)

In [20]:
df[["started_at", "ended_at"]].describe()

,started_at,ended_at
count,10000,10000
mean,2026-08-16 13:30:21.506968064,2026-08-16 13:41:06.682469632
min,2026-07-31 23:14:12.623000,2026-08-01 00:00:44.341000
25%,2026-08-08 19:35:08.180250112,2026-08-08 19:42:31.776750080
50%,2026-08-16 08:11:09.753499904,2026-08-16 08:19:02.511000064
75%,2026-08-24 10:25:37.163000064,2026-08-24 10:33:02.210749952
max,2026-08-31 23:44:40.435000,2026-08-31 23:55:19.465000


In [21]:
df["duration_minutes"] = (
    df["ended_at"]
    - df["started_at"]
).dt.total_seconds() / 60

In [22]:
df["duration_minutes"].describe()

count    10000.000000
mean        10.752925
std         24.509227
min          1.013683
25%          4.454838
50%          6.681183
75%         11.110962
max        997.025633
Name: duration_minutes, dtype: float64

In [23]:
df[
    df["duration_minutes"] <= 0
].head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_minutes


In [24]:
df[
    df["duration_minutes"] > 24 * 60
].head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_minutes
